# Hướng dẫn chạy Legal QA API Server trên Kaggle với Cloudflare Tunnel

Notebook này giúp bạn host toàn bộ backend (gồm model ML và logic RAG) lên Kaggle và public API qua mạng Internet bằng Cloudflare Tunnel. Sau đó Frontend React ở máy local của bạn có thể gọi tới đường dẫn này.

**Lưu ý quan trọng trước khi chạy:**
1. Hãy bật **GPU (P100 hoặc T4x2)** trong cài đặt Notebook (Session options -> Accelerator).
2. Bạn cần đưa toàn bộ source code `legal-qa-vn` (bao gồm code, thư mục `chroma_db`, `models` nếu có) vào Kaggle. Bạn có thể nén thành file ZIP, upload lên phần **Data** (Add Input) của Kaggle.
3. Nếu source code nằm ở Input, bạn cần copy nó vào thư mục `/kaggle/working` vì thư mục Input là Read-only (ChromaDB cần quyền read-write ở một số file lock).

In [1]:
# !rm -rf /kaggle/working/legal-qa-vn

In [2]:
# !rm -rf /kaggle/working/legal-qa-vn/chroma_db

In [3]:
!git clone https://github.com/PTIT-D22KH02-PDDT/legal-qa-vn -b feat/duong/ui

Cloning into 'legal-qa-vn'...
remote: Enumerating objects: 1424, done.
remote: Counting objects: 100% (529/529), done.
remote: Compressing objects: 100% (333/333), done.
remote: Total 1424 (delta 262), reused 351 (delta 191), pack-reused 895 (from 1)
Receiving objects: 100% (1424/1424), 2.18 MiB | 12.05 MiB/s, done.
Resolving deltas: 100% (776/776), done.


In [4]:
cd legal-qa-vn

/kaggle/working/legal-qa-vn


In [5]:
!git pull

Already up to date.


In [6]:
cp -r /kaggle/input/datasets/sandaria/5-5-2026/chroma_db /kaggle/working/legal-qa-vn

In [7]:
# !unzip /kaggle/input/notebooks/vucongtuanduong/run-nlp/shard7.zip -d /kaggle/working/legal-qa-vn/shard7

In [8]:
# !cp -r /kaggle/working/legal-qa-vn/shard7/kaggle/working/shard7 /kaggle/working/legal-qa-vn/chroma_db

In [9]:
%%capture
!pip install uv
# Xóa bỏ hoàn toàn pywin32 trong pyproject.toml khi chạy trên Kaggle/Linux để tránh lỗi PEP 508
!uv add fastapi uvicorn httpx python-dotenv bitsandbytes accelerate
!uv sync

In [10]:
# # 2. Cài đặt các thư viện cần thiết
# %%capture
# !pip install -q fastapi uvicorn sentence-transformers transformers torch httpx pydantic python-dotenv chromadb

In [11]:
# 3. Tải và cấp quyền cho Cloudflared (để tạo Tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print("Cloudflared downloaded.")

Cloudflared downloaded.


In [12]:
# !uv run /kaggle/working/legal-qa-vn/migrations/add_trang_thai_chroma.py

INFO: Collection 'legal_documents' có 73633 chunks. Bắt đầu migration ...
INFO: Tiến độ: 500/73633 chunks đã xử lý (updated=500, skipped=0).
INFO: Tiến độ: 1000/73633 chunks đã xử lý (updated=1000, skipped=0).
INFO: Tiến độ: 1500/73633 chunks đã xử lý (updated=1500, skipped=0).
INFO: Tiến độ: 2000/73633 chunks đã xử lý (updated=2000, skipped=0).
INFO: Tiến độ: 2500/73633 chunks đã xử lý (updated=2500, skipped=0).
INFO: Tiến độ: 3000/73633 chunks đã xử lý (updated=3000, skipped=0).
INFO: Tiến độ: 3500/73633 chunks đã xử lý (updated=3500, skipped=0).
INFO: Tiến độ: 4000/73633 chunks đã xử lý (updated=4000, skipped=0).
INFO: Tiến độ: 4500/73633 chunks đã xử lý (updated=4500, skipped=0).
INFO: Tiến độ: 5000/73633 chunks đã xử lý (updated=5000, skipped=0).
INFO: Tiến độ: 5500/73633 chunks đã xử lý (updated=5500, skipped=0).
INFO: Tiến độ: 6000/73633 chunks đã xử lý (updated=6000, skipped=0).
INFO: Tiến độ: 6500/73633 chunks đã xử lý (updated=6500, skipped=0).
INFO: Tiến độ: 7000/73633 chunk

In [13]:
# !uv run /kaggle/working/legal-qa-vn/migrations/add_trang_thai_sqlite.py

ERROR: Không tìm thấy file database: database/legal_documents.db


In [ ]:
import subprocess
import threading
import time
import re

def log_reader(pipe, prefix):
    try:
        for line in iter(pipe.readline, ''):
            if line:
                print(f"[{prefix}] {line.strip()}")
    except Exception:
        pass

print("Starting Servers...")
# Chạy host1.py (bản tối ưu 4-bit) để tránh lỗi OOM
host_proc = subprocess.Popen(["uv", "run", "python", "scripts/host1.py", "--host", "127.0.0.1", "--port", "8000"], 
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
threading.Thread(target=log_reader, args=(host_proc.stdout, "HOST"), daemon=True).start()

print("Waiting 45s for models to load (4-bit mode)...")
time.sleep(45)

api_proc = subprocess.Popen(["uv", "run", "python", "api_server.py"], 
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
threading.Thread(target=log_reader, args=(api_proc.stdout, "API"), daemon=True).start()

print("Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8001"], 
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_found = False
for line in iter(tunnel_proc.stdout.readline, ''):
    if not url_found and "trycloudflare.com" in line:
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
        if match:
            print("\n" + "="*60)
            print("🎉 YOUR PUBLIC API URL:", match.group(1))
            print("="*60 + "\n")
            url_found = True
    print(f"[TUNNEL] {line.strip()}")

Starting Servers...
Waiting 45s for models to load (4-bit mode)...
[HOST] /kaggle/working/legal-qa-vn/scripts/host1.py:267: DeprecationWarning:
[HOST] on_event is deprecated, use lifespan event handlers instead.
[HOST] 
[HOST] Read more about it in the
[HOST] [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
[HOST] 
[HOST] @app.on_event("startup")
[HOST] INFO:     Started server process [284]
[HOST] INFO:     Waiting for application startup.
[HOST] `torch_dtype` is deprecated! Use `dtype` instead!
[HOST] The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Starting Cloudflare Tunnel...
[TUNNEL] 2026-05-10T15:34:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee,